# 🔥 Wildfire Detection - Global Solution FIAP 2026
Este notebook implementa duas CNNs (SimpleFireNet e DeepFireNet) para classificar imagens de satélite em **wildfire** (queimada ativa) ou **nowildfire**.

O notebook está organizado para rodar do zero, baixando automaticamente os dados e as dependências necessárias.

## 1. Instalação de dependências

In [ ]:
!pip install -q torch torchvision gradio kaggle matplotlib pillow scikit-learn

## 2. Montar Google Drive (opcional)
Caso queira salvar os modelos treinados e os resultados, monte o Drive. Se preferir rodar tudo localmente no Colab, pode pular.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Pasta onde os modelos e gráficos serão salvos (criada automaticamente)
SAVE_DIR = "/content/drive/MyDrive/GS_ACV"
import os
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Arquivos serão salvos em: {SAVE_DIR}")

## 3. Download do dataset do Kaggle
Primeiro, configure sua chave da API do Kaggle. Você precisa ter um arquivo `kaggle.json` com suas credenciais.

**Instruções:**
1. Crie uma conta no Kaggle.
2. Vá em Settings → API → Create New API Token.
3. Faça upload do arquivo `kaggle.json` aqui no Colab ou cole o conteúdo nas variáveis abaixo.

In [ ]:
# Substitua com seu username e key do Kaggle
kaggle_username = "SEU_USERNAME"   # ← altere aqui
kaggle_key = "SUA_KEY"            # ← altere aqui

# Configuração do Kaggle
!mkdir -p ~/.kaggle
import json
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
!chmod 600 /root/.kaggle/kaggle.json

# Download do dataset
!kaggle datasets download -d abdelghaniaaba/wildfire-prediction-dataset
!unzip -q wildfire-prediction-dataset.zip -d datas2/
!find datas2 -type d | head -5

## 4. Carregar dados e definir transformações

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {DEVICE}")

# Caminhos
TRAIN_DIR = "/content/datas2/train"
VAL_DIR   = "/content/datas2/valid"
TEST_DIR  = "/content/datas2/test"

IMG_SIZE = 224
BATCH_SIZE = 32

# Transformações
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_test_transforms)
test_dataset  = datasets.ImageFolder(TEST_DIR,  transform=val_test_transforms)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

CLASS_NAMES = train_dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes: {CLASS_NAMES}")
print(f"Treino: {len(train_dataset)} imagens")
print(f"Validação: {len(val_dataset)} imagens")
print(f"Teste: {len(test_dataset)} imagens")

## 5. Definição dos modelos

In [ ]:
import torch.nn as nn

class SimpleFireNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 56 * 56, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

class DeepFireNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

# Verificação rápida
simple = SimpleFireNet(NUM_CLASSES).to(DEVICE)
deep   = DeepFireNet(NUM_CLASSES).to(DEVICE)
dummy = torch.randn(1, 3, 224, 224).to(DEVICE)
print(f"SimpleFireNet parâmetros: {sum(p.numel() for p in simple.parameters()):,}")
print(f"DeepFireNet parâmetros:   {sum(p.numel() for p in deep.parameters()):,}")

## 6. Treinamento (opcional)
Execute esta célula se quiser treinar os modelos do zero. **Atenção:** o treinamento pode levar cerca de 30-40 minutos em GPU. Se preferir usar modelos pré-treinados, pule esta célula e vá para a seção 7.

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time

def treinar_modelo(modelo, nome, epochs=30):
    modelo = modelo.to(DEVICE)
    criterio = nn.CrossEntropyLoss()
    otimizador = optim.Adam(modelo.parameters(), lr=1e-3)
    scheduler = ReduceLROnPlateau(otimizador, patience=3, factor=0.5)

    historico = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    melhor_acc = 0.0
    patience_counter = 0
    EARLY_STOP = 5

    print(f"\n{'='*50}")
    print(f"Treinando: {nome}")
    print(f"{'='*50}")

    for epoch in range(epochs):
        inicio = time.time()

        # Treino
        modelo.train()
        loss_treino, acertos, total = 0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            otimizador.zero_grad()
            saidas = modelo(imgs)
            loss = criterio(saidas, labels)
            loss.backward()
            otimizador.step()

            loss_treino += loss.item()
            acertos += (saidas.argmax(1) == labels).sum().item()
            total += labels.size(0)

        acc_treino = acertos / total
        loss_treino = loss_treino / len(train_loader)

        # Validação
        modelo.eval()
        loss_val, acertos_val, total_val = 0, 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                saidas = modelo(imgs)
                loss = criterio(saidas, labels)
                loss_val += loss.item()
                acertos_val += (saidas.argmax(1) == labels).sum().item()
                total_val += labels.size(0)

        acc_val = acertos_val / total_val
        loss_val = loss_val / len(val_loader)

        historico["train_loss"].append(loss_treino)
        historico["val_loss"].append(loss_val)
        historico["train_acc"].append(acc_treino)
        historico["val_acc"].append(acc_val)

        scheduler.step(loss_val)

        duracao = time.time() - inicio
        print(f"Época {epoch+1:02d}/{epochs} | Loss treino: {loss_treino:.4f} | Acc treino: {acc_treino:.4f} | "
              f"Loss val: {loss_val:.4f} | Acc val: {acc_val:.4f} | {duracao:.0f}s")

        if acc_val > melhor_acc:
            melhor_acc = acc_val
            torch.save(modelo.state_dict(), f"{SAVE_DIR}/{nome}_melhor.pth")
            print(f"  ✓ Melhor modelo salvo (acc val: {melhor_acc:.4f})")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP:
                print(f"  Early stopping na época {epoch+1}")
                break

    print(f"\nMelhor acc validação ({nome}): {melhor_acc:.4f}")
    return historico

# Descomente as linhas abaixo para treinar:
# historico_simple = treinar_modelo(SimpleFireNet(NUM_CLASSES), "SimpleFireNet")
# historico_deep   = treinar_modelo(DeepFireNet(NUM_CLASSES),   "DeepFireNet")

print("Treinamento não foi executado. Caso queira treinar, descomente as linhas acima.")

## 7. Download dos modelos pré-treinados (alternativa)
Se você não treinou os modelos na seção anterior, pode baixar os pesos já treinados disponíveis no Hugging Face.

In [ ]:
HF_SPACE = "https://huggingface.co/spaces/pemenezzz/wildfire-detection/tree/main"
MODELOS = ["SimpleFireNet_melhor.pth", "DeepFireNet_melhor.pth"]

for modelo in MODELOS:
    destino = f"{SAVE_DIR}/{modelo}"
    if not os.path.exists(destino):
        print(f"Baixando {modelo}...")
        !wget -q "{HF_SPACE}/{modelo}" -O "{destino}"
        print(f"  ✓ Salvo em {destino}")
    else:
        print(f"  ✓ {modelo} já existe, pulando download.")

## 8. Avaliação dos modelos no conjunto de teste

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import matplotlib.pyplot as plt

def avaliar_modelo(ModelClass, nome_arquivo, nome):
    modelo = ModelClass(NUM_CLASSES).to(DEVICE)
    caminho = f"{SAVE_DIR}/{nome_arquivo}_melhor.pth"
    if not os.path.exists(caminho):
        print(f"Arquivo {caminho} não encontrado. Execute o treinamento ou o download dos modelos.")
        return None, None, None

    modelo.load_state_dict(torch.load(caminho, map_location=DEVICE))
    modelo.eval()

    todas_preds, todos_labels = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(DEVICE)
            saidas = modelo(imgs)
            preds = saidas.argmax(1).cpu().numpy()
            todas_preds.extend(preds)
            todos_labels.extend(labels.numpy())

    todas_preds = np.array(todas_preds)
    todos_labels = np.array(todos_labels)
    acc = (todas_preds == todos_labels).mean()

    print(f"\n{'='*50}")
    print(f"Resultado no TEST SET — {nome}")
    print(f"{'='*50}")
    print(f"Acurácia: {acc:.4f} ({acc*100:.2f}%)")
    print(f"\nRelatório por classe:")
    print(classification_report(todos_labels, todas_preds, target_names=CLASS_NAMES))

    # Matriz de confusão
    cm = confusion_matrix(todos_labels, todas_preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES, rotation=15)
    ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.set_title(f"Matriz de Confusão — {nome}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max()/2 else "black",
                    fontsize=14, fontweight="bold")
    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/confusion_{nome}.png", dpi=150)
    plt.show()

    return acc, todas_preds, todos_labels

# Avaliar ambos os modelos (se disponíveis)
acc_simple, _, _ = avaliar_modelo(SimpleFireNet, "SimpleFireNet", "SimpleFireNet")
acc_deep, _, _   = avaliar_modelo(DeepFireNet,   "DeepFireNet",   "DeepFireNet")

if acc_simple is not None and acc_deep is not None:
    print(f"\nComparação final: SimpleFireNet {acc_simple*100:.2f}% | DeepFireNet {acc_deep*100:.2f}%")
    if acc_simple > acc_deep:
        print("Melhor modelo: SimpleFireNet")
    else:
        print("Melhor modelo: DeepFireNet")

## 9. Interface Gradio para demonstração
A célula abaixo cria uma interface web onde você pode fazer upload de uma imagem de satélite e obter a classificação.

In [ ]:
import gradio as gr
from PIL import Image

# Carrega o melhor modelo (SimpleFireNet)
caminho_modelo = f"{SAVE_DIR}/SimpleFireNet_melhor.pth"
if os.path.exists(caminho_modelo):
    modelo_final = SimpleFireNet(NUM_CLASSES).to(DEVICE)
    modelo_final.load_state_dict(torch.load(caminho_modelo, map_location=DEVICE))
    modelo_final.eval()
    print("Modelo carregado com sucesso!")
else:
    raise FileNotFoundError(f"Modelo não encontrado em {caminho_modelo}. Execute o download ou treinamento primeiro.")

def classificar(imagem):
    img = val_test_transforms(imagem).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        saidas = modelo_final(img)
        probs = torch.softmax(saidas, dim=1)[0]
    return {CLASS_NAMES[0]: float(probs[0]), CLASS_NAMES[1]: float(probs[1])}

app = gr.Interface(
    fn=classificar,
    inputs=gr.Image(type="pil", label="Imagem de satélite"),
    outputs=gr.Label(num_top_classes=2, label="Classificação"),
    title="🔥 Wildfire Detection — SimpleFireNet",
    description="Classifica imagens de satélite em **wildfire** (queimada ativa) ou **nowildfire**. Projeto Global Solution FIAP 2026.",
)

app.launch(share=True)